# 20.3 Experiment tracking & reproducibility

An experiment tracker is the memory of learning: it records data, code, config, seed, metric, and artifact so a result can be believed twice.

Validation metrics feed run selection, while randomness and optimization feed the variance we must record rather than explain away.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(203)


def summarize_runs(workload):
    gap = np.abs(workload["rerun_metric"] - workload["recorded_metric"])
    load = int(workload["recorded_metric"].size)
    seconds = max(float(workload["duration_seconds"]), 1.0)
    return {
        "name": workload["name"],
        "requests": load,
        "p50_ms": float(np.percentile(workload["tracking_latency_ms"], 50)),
        "p95_ms": float(np.percentile(workload["tracking_latency_ms"], 95)),
        "throughput_qps": load / seconds,
        "cost_dollars": float(np.sum(workload["cost_usd"])),
        "drift_stat": float(np.std(workload["recorded_metric"])),
        "reproducibility_gap": float(np.mean(gap)),
    }


def simulate_run_log(name, run_count, seed_variance=0.002, stale_rate=0.0, config_drift_rate=0.0):
    base_quality = RNG.normal(0.817, 0.018, run_count)
    seed_noise = RNG.normal(0.0, seed_variance, run_count)
    recorded_metric = base_quality + seed_noise
    rerun_metric = base_quality + RNG.normal(0.0, seed_variance, run_count)
    stale = RNG.random(run_count) < stale_rate
    config_drift = RNG.random(run_count) < config_drift_rate
    rerun_metric[stale] -= RNG.uniform(0.006, 0.025, int(np.sum(stale)))
    rerun_metric[config_drift] += RNG.normal(0.0, 0.015, int(np.sum(config_drift)))
    tracking_latency_ms = RNG.lognormal(mean=np.log(40.0), sigma=0.40, size=run_count)
    cost_usd = 0.002 + 0.000002 * tracking_latency_ms
    return {
        "name": name,
        "recorded_metric": recorded_metric,
        "rerun_metric": rerun_metric,
        "tracking_latency_ms": tracking_latency_ms,
        "cost_usd": cost_usd,
        "duration_seconds": max(run_count / 2.0, 1.0),
        "stale": stale,
        "config_drift": config_drift,
    }


def make_workload_ladder():
    d1 = {
        "name": "D1 three hand-entered runs",
        "recorded_metric": np.array([0.811, 0.824, 0.817]),
        "rerun_metric": np.array([0.810, 0.823, 0.816]),
        "tracking_latency_ms": np.array([28.0, 31.0, 35.0]),
        "cost_usd": np.array([0.002, 0.002, 0.002]),
        "duration_seconds": 3.0,
        "stale": np.array([False, False, False]),
        "config_drift": np.array([False, False, False]),
    }
    d2 = simulate_run_log("D2 100 clean tracked runs", 100, seed_variance=0.001)
    d3 = simulate_run_log("D3 seed variance plus stale data", 1_000, seed_variance=0.006, stale_rate=0.08, config_drift_rate=0.04)
    d4 = simulate_run_log("D4 real-ish experiment log", 20_000, seed_variance=0.004, stale_rate=0.03, config_drift_rate=0.02)
    d5 = simulate_run_log("D5 production-scale registry log", 100_000, seed_variance=0.005, stale_rate=0.04, config_drift_rate=0.03)
    return [d1, d2, d3, d4, d5]


## The concept, built once: immutable run records

The lesson identity is $r=(d,c,\theta,s,m,a)$: data version, code, config, seed, metric, and artifact. We implement the record and assert that the tuple has 6 components.

In [ ]:

def record_run(data_version, code, config, seed, metric, artifact):
    return {
        "data_version": data_version,
        "code": code,
        "config": dict(config),
        "seed": int(seed),
        "metric": float(metric),
        "artifact": artifact,
    }

run = record_run("data-v1", "abc123", {"lr": 0.01}, 7, 0.817, "model.pkl")
run_tuple = (run["data_version"], run["code"], run["config"], run["seed"], run["metric"], run["artifact"])
assert len(run_tuple) == 6
print(run_tuple)


Run selection uses recorded metrics, not memory. The lesson's seed scores $0.811,0.824,0.817$ have mean about $0.817$, and the losses $0.42,0.39,0.44$ select $0.39$.

In [ ]:

seed_scores = np.array([0.811, 0.824, 0.817])
losses = np.array([0.42, 0.39, 0.44])
recorded = 0.817
rerun = 0.816
mean_score = float(np.mean(seed_scores))
best_loss = float(np.min(losses))
gap = abs(rerun - recorded)
assert np.isclose(mean_score, 0.8173333333333334)
assert np.isclose(best_loss, 0.39)
assert np.isclose(gap, 0.001)
print("mean seed score", round(mean_score, 3))
print("best loss", best_loss)
print("reproducibility gap", gap)


## The dataset ladder

Family F17 has no shared ML-training ladder here. We build the operational workload ladder inline: D1 tiny trace, D2 larger volume, D3 spikes or drift, D4 real-ish synthetic trace, and D5 production-scale simulation.

In [ ]:

workloads = make_workload_ladder()
for workload in workloads:
    print(workload["name"])
    print("shape", workload["recorded_metric"].shape)
    print("sample metrics", np.round(workload["recorded_metric"][:5], 4))


## Run the same method across D1-D5

The metric is reproducibility gap $|m_{rerun}-m_{recorded}|$. The table keeps operational latency, cost, throughput, and seed-spread drift visible alongside the scientific gap.

In [ ]:

workloads = make_workload_ladder()
summaries = [summarize_runs(workload) for workload in workloads]

header = f"{'rung':<42} {'load':>10} {'p50':>9} {'p95':>9} {'cost':>10} {'qps':>10} {'reproducibility_gap':>12}"
print(header)
for row in summaries:
    line = f"{row['name']:<42} {row['requests']:>10} {row['p50_ms']:>9.2f} {row['p95_ms']:>9.2f} {row['cost_dollars']:>10.3f} {row['throughput_qps']:>10.2f} {row['reproducibility_gap']:>12.5f}"
    print(line)


The first row is the operational artifact: p95 latency, total cost, and throughput by rung. The second plot is the lesson metric versus load.

In [ ]:

names = [row["name"].split()[0] for row in summaries]
loads = np.array([row["requests"] for row in summaries], dtype=float)
p95_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_values = np.array([row["cost_dollars"] for row in summaries], dtype=float)
throughput_values = np.array([row["throughput_qps"] for row in summaries], dtype=float)
metric_values = np.array([row["reproducibility_gap"] for row in summaries], dtype=float)
cost_per_request = cost_values / loads
normalized_p95 = p95_values / max(float(np.max(p95_values)), 1.0)
normalized_cost = cost_per_request / max(float(np.max(cost_per_request)), 1.0)
normalized_throughput = throughput_values / max(float(np.max(throughput_values)), 1.0)

fig = plt.figure(figsize=(15, 7))
grid = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.15])
for index, name in enumerate(names):
    ax = fig.add_subplot(grid[0, index])
    values = [normalized_p95[index], normalized_cost[index], normalized_throughput[index]]
    ax.bar(["p95", "cost", "qps"], values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("normalized")
summary_ax = fig.add_subplot(grid[1, :])
summary_ax.plot(loads, metric_values, marker="o")
summary_ax.set_xscale("log")
summary_ax.set_title("reproducibility gap vs load")
summary_ax.set_xlabel("records or requests")
summary_ax.set_ylabel("reproducibility gap")
fig.suptitle("Per-workload latency, cost, throughput, and metric-vs-load")
fig.tight_layout()


## Pitfall on D5: ignoring seed variance

A one-seed leaderboard can crown a false winner. The fix is to compare paired multi-seed summaries so the mean and spread, not a lucky run, become the reproducible object.

In [ ]:

d5 = workloads[-1]
run_count = d5["recorded_metric"].size
candidate_a = d5["recorded_metric"][: run_count // 2]
candidate_b = d5["recorded_metric"][run_count // 2:]
one_seed_winner = "A" if candidate_a[0] > candidate_b[0] else "B"
mean_winner = "A" if float(np.mean(candidate_a)) > float(np.mean(candidate_b)) else "B"
spread_a = float(np.std(candidate_a))
spread_b = float(np.std(candidate_b))
print("one-seed winner", one_seed_winner)
print("multi-seed mean winner", mean_winner)
print("seed spreads", round(spread_a, 4), round(spread_b, 4))


## Evaluate it + Practice

- Metric: reproducibility gap; no-skill baseline is a spreadsheet row with metric but no data, code, config, seed, or artifact.
- Sanity check: rerunning the same tuple should stay within the stated tolerance.
- Ablation: drop seed from the tuple and observe that variance becomes untraceable.
- Failure signals: stale data flags, config drift, or winners that flip under paired seeds.

Practice: add a new config key and explain why it creates a different run identity.

Practice: compute the 95th percentile reproducibility gap on D5.

Practice: compare two candidates using five paired seed slices.